In [2]:
# IMPORT LIBRARIES
import os
import time
import numpy as np
import faiss

from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader
from google import genai

# LOADING GEMINI API KEY FROM .ENV
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY not found. "
        "Make sure your .env file contains GEMINI_API_KEY=your_key"
    )

print("Gemini API key loaded successfully.")



#INITIALIZE GEMINI CLIENT


client = genai.Client(api_key=api_key)

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.6-flash"

print("Gemini client initialized.")
print("Embedding model:", EMBEDDING_MODEL)
print("Generation model:", GENERATION_MODEL)


# LOAD PDF

PDF_PATH = "sample_data/sample.pdf"

if not Path(PDF_PATH).exists():
    raise FileNotFoundError(
        f"PDF not found at: {PDF_PATH}\n"
        "Place your PDF inside the sample_data folder."
    )

reader = PdfReader(PDF_PATH)

print("PDF loaded successfully.")
print("Number of pages:", len(reader.pages))

# 6. EXTRACT TEXT FROM PDF

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text and page_text.strip():

        pages.append({
            "page": page_number,
            "text": page_text.strip()
        })


print("Pages containing text:", len(pages))

total_characters = sum(
    len(page["text"])
    for page in pages
)

print("Total characters extracted:", total_characters)


# TEXT CHUNKING

def create_chunks(
    pages,
    chunk_size=1000,
    overlap=150
):
    """
    Split PDF text into chunks while keeping page information.

    Each chunk contains:
    - chunk_id
    - page number
    - text
    """

    chunks = []
    chunk_id = 0

    for page_data in pages:

        page_number = page_data["page"]
        text = page_data["text"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end].strip()

            if chunk_text:

                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            start += chunk_size - overlap

    return chunks


chunks = create_chunks(
    pages,
    chunk_size=1000,
    overlap=150
)

print("Number of chunks:", len(chunks))

# DISPLAY SAMPLE CHUNKS

for chunk in chunks[:3]:

    print("\n" + "=" * 70)
    print("Chunk ID:", chunk["chunk_id"])
    print("Page:", chunk["page"])
    print("-" * 70)
    print(chunk["text"][:500])


# GENERATE GEMINI EMBEDDINGS

def generate_embeddings(
    texts,
    batch_size=20,
    max_retries=5
):
    """
    Generate Gemini embeddings in batches.

    Batching reduces the number of API requests.
    Retry logic handles temporary 503 errors.
    """

    all_embeddings = []

    for start in range(0, len(texts), batch_size):

        batch = texts[start:start + batch_size]

        for attempt in range(max_retries):

            try:

                response = client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch
                )

                batch_embeddings = [
                    embedding.values
                    for embedding in response.embeddings
                ]

                all_embeddings.extend(batch_embeddings)

                print(
                    f"Embedded "
                    f"{min(start + batch_size, len(texts))}"
                    f"/{len(texts)} chunks"
                )

                break

            except Exception as e:

                print(
                    f"Embedding attempt "
                    f"{attempt + 1} failed: {e}"
                )

                if attempt < max_retries - 1:

                    wait_time = 2 ** attempt

                    print(
                        f"Retrying in "
                        f"{wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                else:
                    raise

    return all_embeddings


chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = generate_embeddings(
    chunk_texts
)

print("\nEmbedding generation complete.")
print("Chunks:", len(chunks))
print("Embeddings:", len(embeddings))


# CONVERT EMBEDDINGS TO NUMPY ARRAY

embedding_matrix = np.array(
    embeddings,
    dtype="float32"
)

print("Embedding matrix shape:")
print(embedding_matrix.shape)


# CREATE FAISS VECTOR DATABASE

embedding_dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(
    embedding_dimension
)

index.add(embedding_matrix)

print("FAISS vector database created.")
print("Vectors stored:", index.ntotal)


# SEMANTIC SEARCH / RETRIEVAL

def retrieve_chunks(
    question,
    k=3
):
    """
    Convert the question into an embedding
    and retrieve the most similar document chunks.
    """

    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question
    )

    query_embedding = np.array(
        [response.embeddings[0].values],
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    retrieved_chunks = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        retrieved_chunks.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distance)
        })

    return retrieved_chunks


# TEST RETRIEVAL

test_question = "What is the main purpose of this document?"

retrieved = retrieve_chunks(
    test_question,
    k=3
)

for result in retrieved:

    print("\n" + "=" * 70)
    print("Chunk:", result["chunk_id"])
    print("Page:", result["page"])
    print("Distance:", result["distance"])
    print("-" * 70)
    print(result["text"])


# CREATE RAG PROMPT
def create_prompt(
    question,
    retrieved_chunks
):
    """
    Build the prompt containing:
    - Retrieved context
    - User question
    - Instructions for Gemini
    """

    context_parts = []

    for chunk in retrieved_chunks:

        context_parts.append(
            f"[Page {chunk['page']}]\n"
            f"{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a helpful question-answering assistant.

Your task is to answer the user's question using ONLY
the information provided in the context.

Rules:
1. Use only the provided context.
2. Do not use outside knowledge.
3. Do not make up or hallucinate information.
4. If the answer is not present in the context,
   say: "I couldn't find the answer in the provided document."
5. Give a clear and concise answer.
6. When possible, mention the relevant page number.

Context:
==================================================

{context}

==================================================

Question:
{question}

Answer:
"""

    return prompt


#  GENERATE ANSWER USING GEMINI

def generate_answer(
    question,
    retrieved_chunks,
    max_retries=5
):
    """
    Send the question + retrieved context
    to Gemini for final answer generation.
    """

    prompt = create_prompt(
        question,
        retrieved_chunks
    )

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=GENERATION_MODEL,
                contents=prompt
            )

            return response.text

        except Exception as e:

            print(
                f"Generation attempt "
                f"{attempt + 1} failed: {e}"
            )

            if attempt < max_retries - 1:

                wait_time = 2 ** attempt

                print(
                    f"Retrying in "
                    f"{wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                raise


# COMPLETE RAG PIPELINE
def ask_question(
    question,
    k=3,
    show_sources=True
):
    """
    Complete RAG pipeline:

    Question
        |
    Query Embedding
        |
    FAISS Similarity Search
        |
    Relevant Chunks
        |
    RAG Prompt
        |
    Gemini
        |
    Final Answer
    """

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(
        question,
        k=k
    )

    # Step 2: Generate answer
    answer = generate_answer(
        question,
        retrieved_chunks
    )

    print("\n" + "=" * 70)
    print("QUESTION")
    print("=" * 70)
    print(question)

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(answer)

    if show_sources:

        print("\n" + "=" * 70)
        print("SOURCES")
        print("=" * 70)

        pages_used = sorted(
            set(
                chunk["page"]
                for chunk in retrieved_chunks
            )
        )

        print(
            "Relevant PDF pages:",
            ", ".join(
                str(page)
                for page in pages_used
            )
        )

    return answer


# TEST THE RAG CHATBOT

question = "What is the main purpose of this document?"

answer = ask_question(
    question,
    k=3
)


# TEST MULTIPLE QUESTIONS
questions = [
    "What is the main purpose of this document?",
    "What are the main topics discussed?",
    "What are the important conclusions?",
]

for question in questions:

    ask_question(
        question,
        k=3
    )

# CONVERSATIONAL RAG CHATBOT WITH CHAT HISTORY

chat_history = []


def format_chat_history(history):
    """
    Convert chat history into text that can be
    provided to Gemini.
    """

    if not history:
        return "No previous conversation."

    history_text = ""

    for message in history:
        history_text += (
            f"{message['role'].capitalize()}: "
            f"{message['content']}\n"
        )

    return history_text


def conversational_rag(question, k=3):
    """
    Conversational RAG pipeline:

    1. Take current question
    2. Include previous conversation history
    3. Retrieve relevant PDF chunks
    4. Send history + retrieved context + question to Gemini
    5. Generate answer
    6. Save question and answer to chat history
    """

    global chat_history

    # --------------------------------------------------------
    # STEP 1: Retrieve relevant chunks from PDF
    # --------------------------------------------------------

    retrieved_chunks = retrieve_chunks(
        question,
        k=k
    )

    # --------------------------------------------------------
    # STEP 2: Format retrieved PDF context
    # --------------------------------------------------------

    context_parts = []

    for chunk in retrieved_chunks:

        context_parts.append(
            f"[Page {chunk['page']}]\n"
            f"{chunk['text']}"
        )

    document_context = "\n\n".join(context_parts)

    # --------------------------------------------------------
    # STEP 3: Get previous conversation history
    # --------------------------------------------------------

    previous_conversation = format_chat_history(
        chat_history
    )

    # --------------------------------------------------------
    # STEP 4: Create conversational RAG prompt
    # --------------------------------------------------------

    prompt = f"""
You are a helpful conversational question-answering assistant.

You answer questions using the provided PDF context.

You also have access to the previous conversation so that
you can understand follow-up questions and references.

IMPORTANT RULES:

1. Use the PDF context as the primary source of information.
2. Do not invent information.
3. Use previous conversation only to understand what the user
   is referring to.
4. If the answer cannot be found in the PDF context, say:
   "I couldn't find the answer in the provided document."
5. Keep answers clear and concise.
6. When possible, mention the relevant PDF page number.

==================================================
PREVIOUS CONVERSATION
==================================================

{previous_conversation}

==================================================
RETRIEVED PDF CONTEXT
==================================================

{document_context}

==================================================
CURRENT USER QUESTION
==================================================

{question}

==================================================
ANSWER
==================================================
"""

    # --------------------------------------------------------
    # STEP 5: Send everything to Gemini
    # --------------------------------------------------------

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt
    )

    answer = response.text

    # --------------------------------------------------------
    # STEP 6: Save conversation to history
    # --------------------------------------------------------

    chat_history.append({
        "role": "user",
        "content": question
    })

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    # --------------------------------------------------------
    # STEP 7: Display answer
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)

    print(answer)

    print("\n" + "=" * 70)
    print("SOURCE PAGES")
    print("=" * 70)

    pages_used = sorted(
        set(
            chunk["page"]
            for chunk in retrieved_chunks
        )
    )

    print(
        ", ".join(
            str(page)
            for page in pages_used
        )
    )

    return answer

Gemini API key loaded successfully.
Gemini client initialized.
Embedding model: gemini-embedding-001
Generation model: gemini-3.6-flash
PDF loaded successfully.
Number of pages: 11
Pages containing text: 11
Total characters extracted: 32613
Number of chunks: 43

Chunk ID: 0
Page: 1
----------------------------------------------------------------------
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or

Chunk ID: 1
Page: 1
----------------------------------------------------------------------
anslation tasks show these models to
be superior in q

In [4]:
# ============================================================
# START CHAT
# ============================================================

print("=" * 70)
print("CONVERSATIONAL RAG CHATBOT")
print("=" * 70)

print("Ask questions about your PDF.")
print("Type 'exit' to quit.")
print("Type 'clear' to clear conversation history.")

while True:

    question = input("\nYou: ").strip()

    # Exit chatbot
    if question.lower() == "exit":

        print("\nChatbot: Goodbye!")
        break

    # Clear conversation history
    if question.lower() == "clear":

        chat_history = []

        print("\nChatbot: Conversation history cleared.")
        continue

    # Ignore empty questions
    if not question:
        continue

    try:

        conversational_rag(
            question,
            k=3
        )

    except Exception as e:

        print("\nChatbot: An error occurred.")
        print("Error:", e)

CONVERSATIONAL RAG CHATBOT
Ask questions about your PDF.
Type 'exit' to quit.
Type 'clear' to clear conversation history.



You:  What is Transformers?



ANSWER
Based on the provided document, the **Transformer** is a neural sequence transduction model that relies entirely on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolutions (Page 2). 

It follows an encoder-decoder architecture consisting of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder (Page 2).

SOURCE PAGES
2, 3



You:  how does it work



ANSWER
Based on the provided document, the Transformer works through an **encoder-decoder architecture** composed of stacks of $N = 6$ identical layers:

* **Encoder:** Each of its 6 layers contains two sub-layers: 
  1. A self-attention mechanism.
  2. A position-wise fully connected feed-forward network (Page 3).

* **Decoder:** Also composed of 6 identical layers. It contains the same two sub-layers as the encoder, plus a **third sub-layer** that performs multi-head attention over the output of the encoder stack. Additionally, the decoder's self-attention sub-layer uses masking to prevent positions from attending to subsequent positions (Page 3).

* **Sub-Layer Processing & Connections:** 
  * Around each sub-layer, there is a residual connection followed by layer normalization, calculated as $\text{LayerNorm}(x + \text{Sublayer}(x))$ (Page 3).
  * All sub-layers and embedding layers produce outputs with a dimension of $d_{\text{model}} = 512$ to facilitate these residual connectio


You:  exit



Chatbot: Goodbye!
